In [1]:
#1.Importing libraries:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.agents import Tool, initialize_agent, AgentType
from langchain.memory import ConversationBufferWindowMemory
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import StructuredTool
from pydantic.v1 import BaseModel

#2.Loading API KEYS:
load_dotenv()
google_api_key=os.getenv('GOOGLE_API_KEY')

#3.LLM:
llm=ChatGoogleGenerativeAI(model='gemini-2.5-flash',temperature=0,convert_system_message_to_human=True)

#4.Tool:
qa_prompt=PromptTemplate.from_template('Answer user queries clearly: {query}')
qa_chain=LLMChain(llm=llm, prompt=qa_prompt)
qa_tool=Tool(
    name='Simple QA',
    func=qa_chain.invoke,
    description='A basic LLM that answers users queries clearly')

search_tool=Tool(
    name='Intermediate Answer',
    func=TavilySearchResults(max_results=1).invoke,
    description='Use this tool to search for information'
)

#5.Defining memory:
memory=ConversationBufferWindowMemory(k=2,memory_key='chat_history',return_messages=True)

In [2]:
#6.Agent wrap:
agent=initialize_agent(
    tools=[qa_tool],
    llm=llm,
    memory=memory,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

print('\nFirst question: ')
response1=agent.invoke({'input':'What is Langchain?'})
print('\nAgent response: ',response1)

print('\nFollow up question: ')
response2=agent.invoke({'input':'What are its application?'})
print('\nAgent response: ',response2)

print('\nAsk again about previous topic: ')
response3=agent.invoke({'input':'Explain it simply again'})
print('\nAgent response: ',response3)

C:\Users\DELL\anaconda3\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 0.2.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  warn_deprecated(



First question: 


> Entering new AgentExecutor chain...
Action: Simple QA
Action Input: What is Langchain?
Observation: {'query': 'What is Langchain?', 'text': 'LangChain is an **open-source framework** designed to simplify the development of applications powered by Large Language Models (LLMs).\n\nIn essence, it\'s a **toolkit or an orchestration layer** that helps developers go beyond simply sending a single prompt to an LLM. It allows you to build complex, multi-step applications by "chaining" together various components, including LLMs themselves, external data sources, and other tools.\n\nHere\'s a breakdown of what LangChain is and why it\'s so useful:\n\n### What Problem Does LangChain Solve?\n\nWhile LLMs are incredibly powerful, they have limitations:\n1.  **Lack of Real-time Data:** They are trained on historical data and don\'t inherently know about current events or specific private information.\n2.  **No Memory:** By default, an LLM treats each prompt as a new interactio

In [3]:
#6.Agent wrap:
agent=initialize_agent(
    tools=[qa_tool],
    llm=llm,
    memory=memory,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

print('\nFirst question: ')
response1=agent.invoke({'input':'What is Langchain?'})
print('\nAgent response: ',response1)

print('\nFollow up question: ')
response2=agent.invoke({'input':'What are its application?'})
print('\nAgent response: ',response2)

print('\nAsk again about previous topic: ')
response3=agent.invoke({'input':'Explain it simply again'})
print('\nAgent response: ',response3)


First question: 


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Simple QA
Action Input: What is Langchain?
Observation: {'query': 'What is Langchain?', 'text': 'LangChain is an **open-source framework** designed to simplify the development of applications powered by Large Language Models (LLMs).\n\nIn essence, it\'s a **toolkit or an orchestration layer** that helps developers go beyond simply sending a single prompt to an LLM. It allows you to build complex, multi-step applications by "chaining" together various components, including LLMs themselves, external data sources, and other tools.\n\nHere\'s a breakdown of what LangChain is and why it\'s so useful:\n\n### What Problem Does LangChain Solve?\n\nWhile LLMs are incredibly powerful, they have limitations:\n1.  **Lack of Real-time Data:** They are trained on historical data and don\'t inherently know about current events or specific private information.\n2.  **No Memory:** By default, an LLM 

In [4]:
#6.Agent wrap:
agent=initialize_agent(
    tools=[search_tool],
    llm=llm,
    memory=memory,
    agent=AgentType.SELF_ASK_WITH_SEARCH,
    verbose=True,
    handle_parsing_errors=True
)

print('\nFirst question: ')
response1=agent.invoke({'input':'What is Langchain?'})
print('\nAgent response: ',response1)

print('\nFollow up question: ')
response2=agent.invoke({'input':'What are its application?'})
print('\nAgent response: ',response2)

print('\nAsk again about previous topic: ')
response3=agent.invoke({'input':'Explain it simply again'})
print('\nAgent response: ',response3)


First question: 


> Entering new AgentExecutor chain...
No.
So the final answer is: Langchain is a framework designed to simplify the creation of applications using large language models (LLMs). It provides tools, components, and interfaces to build LLM-powered applications, allowing developers to chain together different components like LLMs, prompt templates, and agents to create more complex and powerful applications.

> Finished chain.

Agent response:  {'input': 'What is Langchain?', 'chat_history': [HumanMessage(content='What are its application?'), AIMessage(content='LangChain\'s applications are quite broad, stemming from its ability to connect LLMs with external data, memory, and tools. Here are some key applications:\n\n*   **Question Answering over Custom Data (RAG - Retrieval Augmented Generation):** This is one of the most popular uses. LangChain allows you to connect an LLM to your own documents, databases, or websites. The LLM can then retrieve relevant information fro

In [5]:
#Tool:
qa_tool_structured=Tool(
    name="SimpleQA",
    func=lambda q: qa_chain.invoke({"query": q})["text"],
    description="Use this tool to answer general questions clearly."
)

#6.Agent wrap:
agent=initialize_agent(
    tools=[qa_tool_structured],
    llm=llm,
    memory=memory,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

print('\nFirst question: ')
response1=agent.invoke({'input':'What is Langchain?'})
print('\nAgent response: ',response1['output'])

print('\nFollow up question: ')
response2=agent.invoke({'input':'What are its application?'})
print('\nAgent response: ',response2['output'])

print('\nAsk again about previous topic: ')
response3=agent.invoke({'input':'Explain it simply again'})
print('\nAgent response: ',response3['output'])


First question: 


> Entering new AgentExecutor chain...
Action:
```json
{
  "action": "SimpleQA",
  "action_input": "What is Langchain?"
}
```

Observation: **LangChain is an open-source framework designed to simplify the development of applications powered by Large Language Models (LLMs).**

In essence, it acts as a toolkit that allows developers to connect LLMs with external data sources, computation, and other tools, enabling them to build more complex and powerful applications than a standalone LLM could achieve.

Here's a breakdown of what LangChain is and why it's important:

### What Problem Does LangChain Solve?

While LLMs are incredibly powerful, they have limitations:
1.  **Lack of Up-to-Date Information:** Their knowledge is limited to their training data.
2.  **Inability to Perform Actions:** They can't directly interact with the real world (e.g., search the web, run code, access databases).
3.  **Limited Context Window:** They can only process a certain amount of text a